In [ ]:
%load_ext autoreload
%autoreload 2

In [57]:
import warnings

from embedding.hybrid_search import HybridSearch

warnings.filterwarnings("ignore")

from collections import defaultdict
from utils.constants import BATCH_SIZE, NEAREST_NEIGHBOUR_COUNT
from tqdm import tqdm
from pathlib import Path
from chunking import chunk_files
from embedding import FAISS, HybridSearch, BM25_Plus, ReciprocalRankFusion, SearchResult, DocumentStore
from evaluator import Evaluator, SearchQueryData
from utils.embed_input import datadict2embed_input, jsonl2datadict
from pprint import pprint
import json

In [84]:
doc_store = DocumentStore()
dense = FAISS(batch_size=BATCH_SIZE)
bm25 = BM25_Plus()

curr_dir = Path.cwd().parent / "data" / "test-repo"
print(curr_dir)
with open("output.jsonl", "w") as f:
    chunk_files(curr_dir, f)

with open("output.jsonl", "r") as f:
    lines = f.readlines()
    n_chunks = len(lines)
    metadatas = list(map(jsonl2datadict, lines))

target: list[SearchQueryData] = []
with open("../rag_eval_dataset_symbolic.jsonl", "r") as f:
    for line in f.readlines():
        target.append(json.loads(line))


texts: list[str] = []
chunk_ids: list[str] = []
for i in range(n_chunks):
    embed_input = datadict2embed_input(metadatas[i])
    chunk_ids.append(metadatas[i].chunk_id)
    doc_store.insert(metadatas[i])
    texts.append(embed_input)

hybrid_search = HybridSearch([dense, bm25], ReciprocalRankFusion())
hybrid_search.fit(texts, chunk_ids)

/home/gaurav/coding/SoftwareEngineeringAgent/data/test-repo
Embedding and saving batches: 


100%|██████████| 95/95 [00:02<00:00, 32.15it/s]


In [ ]:
q = ["Helper method to get MongoDB collection for getting books"]

hybrid_result = hybrid_search.search(q, 180)
ids = [res.chunk_id for res in hybrid_result[0]]

In [133]:
doc_store.get(ids)

[ChunkMetaData(chunk_id='21077e26-62f3-4865-872f-44cda1cd9a67', chunk_type='method', name='_get_collection', qualified_name='module.BookController._get_collection', parent_name='BookController', parent_type='class', file_path='backend/src/controllers/book_controller.py', file_name='book_controller.py', module_path='backend.src.controllers.book_controller', language='Python', start_line=27, end_line=32, source_code='def _get_collection(self, collection_name: str):\n        """\n        Helper method to get MongoDB collection.\n        """\n        # self.mongo_db_connection.start_connection()\n        return self.mongo_db_connection.get_collection(collection_name)', parameters=['self', 'collection_name: str'], return_type=None, decorators=[], is_async=False, base_classes=[], imports=[], contains=[], methods=[], contained_in='BookController'),
 ChunkMetaData(chunk_id='3f85ca30-16ff-45cc-858c-bb5e28905119', chunk_type='method', name='_get_collection', qualified_name='module.UserController

In [92]:
ids

['6b1afab0-2663-482b-91ed-681d19ca3ab5',
 'e16c41c7-cae5-4565-ab53-c2c1d0beae04',
 '4b8b9bd5-efba-46df-9f85-be7417960e95',
 '18616ef2-7044-44d1-8d72-8d5db7d84306',
 '3adcab44-c8b2-4ad6-a8d1-7e6b6e5b626f',
 '49b364d6-41c1-4b57-a3e1-5a19f6a07dac',
 '49d9ef58-8e94-4e78-861a-8b3880a31b2c',
 '375be977-c9b9-413d-a259-f4389da76635',
 '851116e4-788f-4f88-b1ea-8b9819f0afcd',
 '6a0e02b7-54da-405b-bdac-a99b486b5e5f',
 '7300b6c6-a3cb-4814-9444-bcdfe7d4f82d',
 '35240b0a-3915-421d-aa33-6a576200d81e',
 '8d7e32a3-ecb2-4509-a515-9cce2ce0c704',
 '0b420832-7a70-413f-b96c-42001d84b41e',
 '288e9c52-51b9-408e-89cb-36d204d95b17',
 '2b1aaf93-3588-4814-834a-b496f6939669',
 '3c9678b9-c0b5-45d2-8ed7-073e5c8244d4',
 '7973cb6e-dacd-4c76-87f7-3a3d264b0c81',
 'c65a175e-3c66-4ff2-94be-361a44e5bf6a',
 '4349b3e0-726c-418d-bd25-e8858fd55876',
 '8aad995a-8256-491e-9fb6-59baa9b3f8ca',
 '6b75cae5-32b0-4c35-b68a-e73beebbadae',
 'e98b4867-4dfe-45ce-9c12-0d386a3c92b9',
 '32d6ffb0-f703-4a97-9ac1-b764687f1eb4',
 '4e14e4da-db47-

In [42]:
len(ids)

5

In [33]:
doc_store.get(ids)

[ChunkMetaData(chunk_id='fcddce00-5fee-43ac-98b2-3b5ed0c53e0e', chunk_type='class', name='UserControllersClass', qualified_name='module.UserControllersClass', parent_name='module', parent_type='module', file_path='backend/src/controllers/user_controller.py', file_name='user_controller.py', module_path='backend.src.controllers.user_controller', language='Python', start_line=15, end_line=421, source_code='class UserControllersClass:\n    def __init__(self):\n        self.Config = Settings()\n        self.mongo_db_connection = MongoDBConnection(self.Config.MONGO_URI, self.Config.DB_NAME)\n        self.email_services = EmailServices()\n        self.password_manager = PasswordManager()\n        self.jwt_manager = JWTManager()\n        self.email_regex_parrern = r"^(?!.*\\.\\.)[a-zA-Z0-9!#$%&\'*+/=?^_`{|}~-]+(?:\\.[a-zA-Z0-9!#$%&\'*+/=?^_`{|}~-]+)*@(?:[a-zA-Z0-9-]+\\.)+[a-zA-Z]{2,}$"\n\n    def _start_connection(self):\n        """\n        Start the MongoDB connection.\n        """\n       